In [3]:
pip install tensorflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 620.7/620.7 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 178.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 203.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 203.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.5/224.5 kB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 8.1 MB/s eta 0:00:00


In [4]:
import tensorflow as tf
from tensorflow.keras.applications import Xception
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import os
import numpy as np

# Set the path to your dataset (where your 'real' and 'fake' face image folders are)
DATASET_DIR = '/content/drive/MyDrive/xceptionnet_data/cropped_faces_2' # Adjust this path for your Kaggle setup
IMAGE_SIZE = 299 # Xception default input size
BATCH_SIZE = 64
EPOCHS = 10

/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:82: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


In [5]:
# Create image data generators
# Xception requires images to be preprocessed with its own function
preprocess_input = tf.keras.applications.xception.preprocess_input

train_datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    preprocessing_function=preprocess_input, # Apply Xception preprocessing
    validation_split=0.2 # Use 20% of data for validation
)

# Load data from directories
train_generator = train_datagen.flow_from_directory(
    DATASET_DIR,
    target_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary', # 'real' or 'fake' (0 or 1)
    subset='training',
    seed=42
)

validation_generator = train_datagen.flow_from_directory(
    DATASET_DIR,
    target_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='validation',
    seed=42
)

# Class labels: {0: 'FAKE', 1: 'REAL'} or vice versa, based on folder names
print("Class Indices:", train_generator.class_indices)

Found 9206 images belonging to 2 classes.
Found 2301 images belonging to 2 classes.
Class Indices: {'FAKE': 0, 'REAL': 1}


In [6]:
# 1. Load the pre-trained Xception model (excluding the final classification layer)
base_model = Xception(
    weights='imagenet',
    include_top=False,
    input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3)
)

# 2. Freeze the weights of the base model layers
for layer in base_model.layers:
    layer.trainable = False

# 3. Add custom classification layers (Head)
x = base_model.output
x = GlobalAveragePooling2D()(x) # Collapse spatial dimensions
x = Dropout(0.5)(x) # Add dropout for regularization
x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x)

# Final output layer for binary classification (REAL or FAKE)
predictions = Dense(1, activation='sigmoid')(x)

# 4. Create the final model
model = Model(inputs=base_model.input, outputs=predictions)

# 5. Compile the model
model.compile(
    optimizer='adam',
    loss='binary_crossentropy', # Appropriate for binary classification
    metrics=['accuracy']
)

model.summary()

83683744/83683744 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 299, 299,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv1        │ (None, 149, 149,  │        864 │ input_layer[0][0] │
│ (Conv2D)            │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv1_bn     │ (None, 149, 149,  │        128 │ block1_conv1[0][… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv1_act    │ (None, 149, 149,  │          0 │ block1_conv1_bn[… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv2        │ (None, 147, 147,  │     18,432 │ block1_conv1_act… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv2_bn     │ (None, 147, 147,  │        256 │ block1_conv2[0][… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv2_act    │ (None, 147, 147,  │          0 │ block1_conv2_bn[… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_sepconv1     │ (None, 147, 147,  │      8,768 │ block1_conv2_act… │
│ (SeparableConv2D)   │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_sepconv1_bn  │ (None, 147, 147,  │        512 │ block2_sepconv1[… │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_sepconv2_act │ (None, 147, 147,  │          0 │ block2_sepconv1_… │
│ (Activation)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_sepconv2     │ (None, 147, 147,  │     17,536 │ block2_sepconv2_… │
│ (SeparableConv2D)   │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_sepconv2_bn  │ (None, 147, 147,  │        512 │ block2_sepconv2[… │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 74, 74,    │      8,192 │ block1_conv2_act… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_pool         │ (None, 74, 74,    │          0 │ block2_sepconv2_… │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 74, 74,    │        512 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 74, 74,    │          0 │ block2_pool[0][0… │
│                     │ 128)              │            │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block3_sepconv1_act │ (None, 74, 74,    │          0 │ add[0][0]       

 Total params: 21,123,881 (80.58 MB)

 Trainable params: 262,401 (1.00 MB)

 Non-trainable params: 20,861,480 (79.58 MB)

In [7]:
print("Starting Model Training...")

# Train the model
history = model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=validation_generator,
    validation_steps=validation_generator.samples // BATCH_SIZE
)

# Save the trained model
model.save('xception_deepfake_detector.h5')
print("Model saved as xception_deepfake_detector.h5")

Starting Model Training...


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
143/143 ━━━━━━━━━━━━━━━━━━━━ 1017s 7s/step - accuracy: 0.7826 - loss: 0.4975 - val_accuracy: 0.8013 - val_loss: 0.4718
Epoch 2/10
  1/143 ━━━━━━━━━━━━━━━━━━━━ 2:00 851ms/step - accuracy: 0.7656 - loss: 0.4797

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:116: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


143/143 ━━━━━━━━━━━━━━━━━━━━ 48s 330ms/step - accuracy: 0.7656 - loss: 0.4797 - val_accuracy: 0.8040 - val_loss: 0.4811
Epoch 3/10
143/143 ━━━━━━━━━━━━━━━━━━━━ 218s 2s/step - accuracy: 0.8131 - loss: 0.4402 - val_accuracy: 0.8022 - val_loss: 0.5076
Epoch 4/10
143/143 ━━━━━━━━━━━━━━━━━━━━ 44s 302ms/step - accuracy: 0.7812 - loss: 0.4495 - val_accuracy: 0.8022 - val_loss: 0.5013
Epoch 5/10
143/143 ━━━━━━━━━━━━━━━━━━━━ 216s 2s/step - accuracy: 0.8230 - loss: 0.4110 - val_accuracy: 0.8134 - val_loss: 0.4476
Epoch 6/10
143/143 ━━━━━━━━━━━━━━━━━━━━ 44s 302ms/step - accuracy: 0.8438 - loss: 0.3463 - val_accuracy: 0.8134 - val_loss: 0.4440
Epoch 7/10
143/143 ━━━━━━━━━━━━━━━━━━━━ 217s 2s/step - accuracy: 0.8375 - loss: 0.3944 - val_accuracy: 0.8147 - val_loss: 0.4324
Epoch 8/10
143/143 ━━━━━━━━━━━━━━━━━━━━ 44s 301ms/step - accuracy: 0.9062 - loss: 0.3315 - val_accuracy: 0.8112 - val_loss: 0.4311
Epoch 9/10
143/143 ━━━━━━━━━━━━━━━━━━━━ 220s 2s/step - accuracy: 0.8370 - loss: 0.3785 - val_accurac

Model saved as xception_deepfake_detector.h5


In [11]:
!pip install opencv-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 59.9 MB/s eta 0:00:00


In [19]:
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
import cv2

# Function to preprocess and predict on a single frame
def predict_deepfake_on_frame(img_path, model_path='xception_deepfake_detector_finetuned.h5'):
    # Load the saved model
    loaded_model = load_model(model_path)

    # Load and resize the image
    img = image.load_img(img_path, target_size=(IMAGE_SIZE, IMAGE_SIZE))
    img_array = image.img_to_array(img)

    # Apply Xception's preprocessing and add a batch dimension
    img_array = preprocess_input(img_array)
    img_array = np.expand_dims(img_array, axis=0)

    # Make prediction
    prediction = loaded_model.predict(img_array)[0][0]

    # The output is a probability between 0 and 1
    if prediction > 0.5:
        return f"FAKE: {prediction:.4f}"
    else:
        return f"REAL: {1 - prediction:.4f}"

# Example Usage (replace 'path/to/test_face.jpg' with an actual face image)
# You would need to extract a face from a video first to use this.
# result = predict_deepfake_on_frame('path/to/test_face.jpg')
# print(f"Prediction: {result}")

In [21]:
from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error
import numpy as np

# Function to evaluate the model on a test dataset
def evaluate_model_on_dataset(test_dir, model_path='xception_deepfake_detector_finetuned.h5'):
    # Load the saved model
    loaded_model = load_model(model_path)

    # Create an image data generator for the test set
    test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

    test_generator = test_datagen.flow_from_directory(
        test_dir,
        target_size=(IMAGE_SIZE, IMAGE_SIZE),
        batch_size=BATCH_SIZE,
        class_mode='binary',
        shuffle=False # Keep the order of images for consistent evaluation
    )

    # Get true labels
    true_labels = test_generator.classes
    class_indices = test_generator.class_indices
    print("Test Class Indices:", class_indices)

    # Get predicted probabilities
    predictions = loaded_model.predict(test_generator)

    # Convert probabilities to binary predictions (0 or 1)
    predicted_labels = (predictions > 0.5).astype(int)

    # Calculate metrics
    accuracy = accuracy_score(true_labels, predicted_labels)
    f1 = f1_score(true_labels, predicted_labels)
    mae = mean_absolute_error(true_labels, predictions) # MAE uses probabilities

    print(f"Accuracy: {accuracy:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print(f"Mean Absolute Error: {mae:.4f}")

# Example Usage (replace '/path/to/test_data' with the actual path to your test data)
evaluate_model_on_dataset('/content/drive/MyDrive/xceptionnet_data/test_videos')

Found 0 images belonging to 0 classes.
Test Class Indices: {}


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


ValueError: The PyDataset has length 0

# Task
Improve the accuracy of the image classification model.

## Fine-tune the base model

### Subtask:
Unfreeze some layers of the pre-trained Xception model and train them along with the new classification layers.


**Reasoning**:
Unfreeze the last 20 layers of the base model and recompile the model.



In [14]:
# Unfreeze the last 20 layers of the base model
unfreeze_layers = 20
for layer in base_model.layers[-unfreeze_layers:]:
    if not isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = True

# Recompile the model
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5), # Use a lower learning rate for fine-tuning
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 299, 299,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv1        │ (None, 149, 149,  │        864 │ input_layer[0][0] │
│ (Conv2D)            │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv1_bn     │ (None, 149, 149,  │        128 │ block1_conv1[0][… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv1_act    │ (None, 149, 149,  │          0 │ block1_conv1_bn[… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv2        │ (None, 147, 147,  │     18,432 │ block1_conv1_act… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv2_bn     │ (None, 147, 147,  │        256 │ block1_conv2[0][… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv2_act    │ (None, 147, 147,  │          0 │ block1_conv2_bn[… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_sepconv1     │ (None, 147, 147,  │      8,768 │ block1_conv2_act… │
│ (SeparableConv2D)   │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_sepconv1_bn  │ (None, 147, 147,  │        512 │ block2_sepconv1[… │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_sepconv2_act │ (None, 147, 147,  │          0 │ block2_sepconv1_… │
│ (Activation)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_sepconv2     │ (None, 147, 147,  │     17,536 │ block2_sepconv2_… │
│ (SeparableConv2D)   │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_sepconv2_bn  │ (None, 147, 147,  │        512 │ block2_sepconv2[… │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 74, 74,    │      8,192 │ block1_conv2_act… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_pool         │ (None, 74, 74,    │          0 │ block2_sepconv2_… │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 74, 74,    │        512 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 74, 74,    │          0 │ block2_pool[0][0… │
│                     │ 128)              │            │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block3_sepconv1_act │ (None, 74, 74,    │          0 │ add[0][0]       

 Total params: 21,123,881 (80.58 MB)

 Trainable params: 7,574,601 (28.89 MB)

 Non-trainable params: 13,549,280 (51.69 MB)

**Reasoning**:
Train the model with the unfrozen layers for additional epochs to fine-tune the model.



In [15]:
print("Starting Model Training...")

# Train the model
history = model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // BATCH_SIZE,
    epochs=EPOCHS * 2, # Train for double the initial epochs
    validation_data=validation_generator,
    validation_steps=validation_generator.samples // BATCH_SIZE
)

# Save the trained model
model.save('xception_deepfake_detector_finetuned.h5')
print("Model saved as xception_deepfake_detector_finetuned.h5")

Starting Model Training...
Epoch 1/20
143/143 ━━━━━━━━━━━━━━━━━━━━ 231s 2s/step - accuracy: 0.8417 - loss: 0.3725 - val_accuracy: 0.8188 - val_loss: 0.4552
Epoch 2/20
  1/143 ━━━━━━━━━━━━━━━━━━━━ 2:12 932ms/step - accuracy: 0.8704 - loss: 0.2947

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:116: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


143/143 ━━━━━━━━━━━━━━━━━━━━ 43s 294ms/step - accuracy: 0.8704 - loss: 0.2947 - val_accuracy: 0.8170 - val_loss: 0.4538
Epoch 3/20
143/143 ━━━━━━━━━━━━━━━━━━━━ 232s 2s/step - accuracy: 0.8563 - loss: 0.3401 - val_accuracy: 0.8228 - val_loss: 0.4116
Epoch 4/20
143/143 ━━━━━━━━━━━━━━━━━━━━ 43s 293ms/step - accuracy: 0.9062 - loss: 0.2442 - val_accuracy: 0.8201 - val_loss: 0.4117
Epoch 5/20
143/143 ━━━━━━━━━━━━━━━━━━━━ 226s 2s/step - accuracy: 0.8661 - loss: 0.3170 - val_accuracy: 0.8397 - val_loss: 0.3809
Epoch 6/20
143/143 ━━━━━━━━━━━━━━━━━━━━ 43s 292ms/step - accuracy: 0.9062 - loss: 0.2557 - val_accuracy: 0.8371 - val_loss: 0.3887
Epoch 7/20
143/143 ━━━━━━━━━━━━━━━━━━━━ 226s 2s/step - accuracy: 0.8696 - loss: 0.3058 - val_accuracy: 0.8433 - val_loss: 0.3760
Epoch 8/20
143/143 ━━━━━━━━━━━━━━━━━━━━ 43s 294ms/step - accuracy: 0.8594 - loss: 0.2751 - val_accuracy: 0.8429 - val_loss: 0.3788
Epoch 9/20
143/143 ━━━━━━━━━━━━━━━━━━━━ 226s 2s/step - accuracy: 0.8822 - loss: 0.2754 - val_accurac

Model saved as xception_deepfake_detector_finetuned.h5


**Reasoning**:
Continue training the model for the remaining epochs to improve accuracy.



In [ ]:
print("Continuing Model Training...")

# Train the model for the remaining epochs
history = model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // BATCH_SIZE,
    epochs=EPOCHS * 2, # Continue for the total number of epochs
    initial_epoch=9, # Start from epoch 10
    validation_data=validation_generator,
    validation_steps=validation_generator.samples // BATCH_SIZE
)

# Save the trained model
model.save('xception_deepfake_detector_finetuned.h5')
print("Model saved as xception_deepfake_detector_finetuned.h5")

Continuing Model Training...
Epoch 10/20
 81/143 ━━━━━━━━━━━━━━━━━━━━ 1:19 1s/step - accuracy: 0.9068 - loss: 0.2113

KeyboardInterrupt: 